# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aneeqahabib/FlyRank_ML_Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

The prediction moment is the end of the previous 30-day window. I therefore use the preceding 30-day counts plus content and keyword metadata that would already exist at that moment. I exclude 90-day performance summaries because they overlap the last-30-day outcome window. Numeric missing values are filled with `0` only after adding `has_*` indicators, so missing keyword or measurement coverage is not silently treated as a real zero. Categorical missing values become an explicit `unknown` level and are one-hot encoded.

In [1]:
import numpy as np
import pandas as pd

DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)
df["is_declining_label"] = df["trend_direction"].eq("down").astype(int)

# These measures end before the last-30-day outcome window.
NUMERIC_FEATURES = [
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "char_count",
    "search_volume",
    "competition",
]
CATEGORICAL_FEATURES = ["content_type", "main_intent", "competition_level"]

feature_input = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
missing_flags = {}
for column in NUMERIC_FEATURES + CATEGORICAL_FEATURES:
    missing_flags[f"has_{column}"] = feature_input[column].notna().astype(int)

numeric_frame = feature_input[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce")
numeric_frame = numeric_frame.replace([np.inf, -np.inf], np.nan).fillna(0)
flag_frame = pd.DataFrame(missing_flags, index=df.index)
categorical_frame = feature_input[CATEGORICAL_FEATURES].fillna("unknown").astype(str)
encoded_categories = pd.get_dummies(
    categorical_frame,
    prefix=CATEGORICAL_FEATURES,
    dtype=float,
)

feature_vector = pd.concat(
    [numeric_frame, flag_frame, encoded_categories],
    axis=1,
).astype(float)
target = df["is_declining_label"].astype(int)

assert len(feature_vector) == len(df)
assert feature_vector.notna().all().all()
assert not set(feature_vector.columns).intersection(
    {"trend_direction", "trend_pct", "is_declining_label", "content_id", "client_id"}
)

print("Rows:", len(feature_vector))
print("Feature-vector columns:", feature_vector.shape[1])
print("Numeric source features:", NUMERIC_FEATURES)
print("Categorical source features:", CATEGORICAL_FEATURES)
print("Target rate:", round(target.mean(), 3))
display(feature_vector.head())

Rows: 30000
Feature-vector columns: 33
Numeric source features: ['impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'days_since_last_update', 'word_count', 'char_count', 'search_volume', 'competition']
Categorical source features: ['content_type', 'main_intent', 'competition_level']
Target rate: 0.542


,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,days_since_last_update,word_count,char_count,search_volume,competition,has_impressions_prev_30d,...,content_type_keyword article,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional,main_intent_unknown,competition_level_HIGH,competition_level_LOW,competition_level_MEDIUM,competition_level_unknown
0,987.0,13.0,9.0,187.0,20.0,3221.0,20457.0,10.0,0.67,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
1,5915.0,1.0,2.0,445.0,25.0,2481.0,15562.0,90.0,0.01,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,6089.0,3.0,3.0,141.0,20.0,3515.0,23643.0,0.0,0.00,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,4206.0,17.0,26.0,463.0,22.0,0.0,0.0,10.0,0.00,1.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,6452.0,2.0,9.0,263.0,14.0,2803.0,17469.0,0.0,0.00,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

The table below records the meaning, missing-value treatment, categorical handling, and availability timing for every source feature. The `has_*` columns are deliberate: the data dictionary says missing keyword and content measurements follow content type, so a missing value can itself carry information about measurement coverage.

In [2]:
feature_notes = pd.DataFrame(
    [
        ["impressions_prev_30d", "Search visibility in days 31-60 before the outcome", "0 if missing; has_ flag", "Yes"],
        ["clicks_prev_30d", "Search clicks in days 31-60 before the outcome", "0 if missing; has_ flag", "Yes"],
        ["sessions_prev_30d", "Analytics sessions in days 31-60 before the outcome", "0 if missing; has_ flag", "Yes, subject to analytics coverage"],
        ["content_age_days", "Days since content creation", "0 if missing; has_ flag", "Yes"],
        ["days_since_last_update", "Days since the last content update", "0 if missing; has_ flag", "Yes"],
        ["word_count", "Measured content word count", "0 if missing plus has_word_count", "Yes"],
        ["char_count", "Measured content character count", "0 if missing plus has_char_count", "Yes"],
        ["search_volume", "Estimated demand for the target keyword", "0 if missing plus has_search_volume", "Yes"],
        ["competition", "Keyword competition estimate from 0 to 1", "0 if missing plus has_competition", "Yes"],
        ["content_type", "Content format", "Missing becomes unknown; one-hot encoded", "Yes"],
        ["main_intent", "Recorded search intent", "Missing becomes unknown; one-hot encoded", "Yes"],
        ["competition_level", "Categorical keyword competition band", "Missing becomes unknown; one-hot encoded", "Yes"],
    ],
    columns=["feature", "meaning", "missing_handling", "available_before_prediction"],
)

print("Feature notes:")
display(feature_notes)
print("\nMissingness is retained by has_* indicators:")
display(df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].isna().mean().mul(100).round(2).rename("missing_percent").to_frame())

Feature notes:


,feature,meaning,missing_handling,available_before_prediction
0,impressions_prev_30d,Search visibility in days 31-60 before the out...,0 if missing; has_ flag,Yes
1,clicks_prev_30d,Search clicks in days 31-60 before the outcome,0 if missing; has_ flag,Yes
2,sessions_prev_30d,Analytics sessions in days 31-60 before the ou...,0 if missing; has_ flag,"Yes, subject to analytics coverage"
3,content_age_days,Days since content creation,0 if missing; has_ flag,Yes
4,days_since_last_update,Days since the last content update,0 if missing; has_ flag,Yes
5,word_count,Measured content word count,0 if missing plus has_word_count,Yes
6,char_count,Measured content character count,0 if missing plus has_char_count,Yes
7,search_volume,Estimated demand for the target keyword,0 if missing plus has_search_volume,Yes
8,competition,Keyword competition estimate from 0 to 1,0 if missing plus has_competition,Yes
9,content_type,Content format,Missing becomes unknown; one-hot encoded,Yes



Missingness is retained by has_* indicators:


,missing_percent
impressions_prev_30d,0.00
clicks_prev_30d,0.00
sessions_prev_30d,0.00
content_age_days,0.00
days_since_last_update,0.00
word_count,25.66
char_count,25.66
search_volume,8.23
competition,8.23
content_type,0.00


## 3. The leakage hunt

I tested three leakage routes. First, `trend_direction` directly defines the label and `trend_pct` is its numeric sibling, so neither can enter the vector. Second, the last-30-day fields and 90-day summaries overlap the label window; they could reveal the outcome and are excluded. Third, IDs and provider/model fields can encode client or production decisions rather than generalizable content signals, so they are reserved for grouping or audit context, not features. The assertions below are the receipt: the deliberately leaky label source reproduces the target exactly, while the selected source features are disjoint from every excluded set.

In [3]:
# The label is directly defined by trend_direction, so both it and trend_pct are suspect.
label_source_columns = {"trend_direction", "trend_pct", "is_declining_label"}
# These windows include the last-30-day outcome window and are not safe inputs.
overlapping_performance_columns = {
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
}
# IDs and model/product provenance are not content signals for this task.
identifier_columns = {"content_id", "client_id"}
product_or_provenance_columns = {"provider_used", "model_used"}

all_excluded_columns = (
    label_source_columns
    | overlapping_performance_columns
    | identifier_columns
    | product_or_provenance_columns
)
source_features_used = set(NUMERIC_FEATURES + CATEGORICAL_FEATURES)

assert source_features_used.isdisjoint(all_excluded_columns)
assert (df["trend_direction"].eq("down").astype(int) == target).all()
assert "trend_direction" not in feature_vector.columns
assert "trend_pct" not in feature_vector.columns

# A deliberate leakage test: the label source reproduces the target exactly.
deliberately_leaky_feature = df["trend_direction"].eq("down").astype(int)
print("Exact label reproduction from trend_direction:", bool((deliberately_leaky_feature == target).all()))
print("Leakage source columns excluded:", sorted(label_source_columns))
print("Overlapping outcome-window columns excluded:", sorted(overlapping_performance_columns))
print("Safe source features used:", sorted(source_features_used))
print("All selected source features pass the exclusion test:", source_features_used.isdisjoint(all_excluded_columns))

Exact label reproduction from trend_direction: True
Leakage source columns excluded: ['is_declining_label', 'trend_direction', 'trend_pct']
Overlapping outcome-window columns excluded: ['ai_traffic_pct', 'avg_position', 'clicks_90d', 'clicks_last_30d', 'ctr', 'engagement_rate', 'impressions_90d', 'impressions_last_30d', 'scroll_rate', 'sessions_90d', 'sessions_last_30d']
Safe source features used: ['char_count', 'clicks_prev_30d', 'competition', 'competition_level', 'content_age_days', 'content_type', 'days_since_last_update', 'impressions_prev_30d', 'main_intent', 'search_volume', 'sessions_prev_30d', 'word_count']
All selected source features pass the exclusion test: True


## 4. What I excluded and why

I excluded label-derived fields, any performance measure whose window overlaps the last-30-day label, identifiers, and production/provenance fields. I also avoid redundant tier columns: the raw safe fields and their missingness indicators are easier to audit. The table below is the compact exclusion register used by the code.

In [4]:
excluded_feature_notes = pd.DataFrame(
    [
        ["trend_direction", "Directly defines is_declining_label; exact label leakage"],
        ["trend_pct", "Numeric sibling of the label definition; outcome-derived"],
        ["impressions_last_30d / clicks_last_30d / sessions_last_30d", "Measured in the outcome window"],
        ["impressions_90d / clicks_90d / sessions_90d", "Trailing totals overlap the outcome window"],
        ["ctr / avg_position / engagement_rate / scroll_rate / ai_traffic_pct", "90-day rates overlap the outcome window"],
        ["content_id / client_id", "Pseudonymous identifiers; grouping only, never predictors"],
        ["provider_used / model_used", "Production/provenance fields, not general content signals"],
        ["age_tier / age_tier_order / freshness_tier / word_count_tier / char_count_tier", "Redundant buckets; raw safe fields are clearer and less duplicated"],
        ["impression_tier / position_tier", "Derived from overlapping 90-day performance fields"],
    ],
    columns=["excluded_fields", "reason"],
)

assert set(excluded_feature_notes["excluded_fields"]).isdisjoint(source_features_used)
print("Excluded fields and reasons:")
display(excluded_feature_notes)
print("\nSelected raw source fields:")
display(feature_notes[["feature", "available_before_prediction"]])

Excluded fields and reasons:


,excluded_fields,reason
0,trend_direction,Directly defines is_declining_label; exact lab...
1,trend_pct,Numeric sibling of the label definition; outco...
2,impressions_last_30d / clicks_last_30d / sessi...,Measured in the outcome window
3,impressions_90d / clicks_90d / sessions_90d,Trailing totals overlap the outcome window
4,ctr / avg_position / engagement_rate / scroll_...,90-day rates overlap the outcome window
5,content_id / client_id,"Pseudonymous identifiers; grouping only, never..."
6,provider_used / model_used,"Production/provenance fields, not general cont..."
7,age_tier / age_tier_order / freshness_tier / w...,Redundant buckets; raw safe fields are clearer...
8,impression_tier / position_tier,Derived from overlapping 90-day performance fi...



Selected raw source fields:


,feature,available_before_prediction
0,impressions_prev_30d,Yes
1,clicks_prev_30d,Yes
2,sessions_prev_30d,"Yes, subject to analytics coverage"
3,content_age_days,Yes
4,days_since_last_update,Yes
5,word_count,Yes
6,char_count,Yes
7,search_volume,Yes
8,competition,Yes
9,content_type,Yes


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.